# Full Pipeline Function Test

Exercises every currently implemented function of chunking
(`src/pipeline/chuncking.py`) → embedding/score preprocessing
(`src/pipeline/embeddings.py` + `src/pipeline/gisting.py`).

Pagination via `paginate_semantic`.

Numbered `00` because it produces no artifacts and is not a step in the
experiment sequence (`01`-`09`) — it's a regression check to run whenever
`src/pipeline/` changes. It reads a committed sample text, so it has no
upstream notebook dependency and can run standalone on a fresh clone.

| Section | Content | Needs API |
|---|---|---|
| §1 | Chunking — plain text adapter, `paginate_semantic` pagination, position-to-chunk lookup, dialogue preservation check | ❌ (mock) / ✅ (real API demo at the end of §1) |
| §2 | Embedding/score preprocessing — intra-chunk importance/similarity flow with mock embeddings | ❌ |
| §3 | Real NIM embedding API — embedding call, similarity score analysis | ✅ `NVIDIA_NIM_API_KEY` |

In [46]:
import os
import sys
from pathlib import Path

# Walk up until src/ is visible, then add the project root to sys.path
# (once importable, the rest of setup — fixing the working directory, loading
# .env, printing whether the API key was picked up — is delegated to
# src.notebook_setup.setup_project())
root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src.notebook_setup import setup_project

setup_project()
API_KEY_SET = bool(os.environ.get("NVIDIA_NIM_API_KEY"))

project root: /Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall
.env 로드: 성공 ✅
NVIDIA_NIM_API_KEY: 설정됨 ✅


## Function index

Every public function this notebook exercises, with where it's demoed.

| Module | Function | Role | Demo location |
|---|---|---|---|
| `chuncking.py` | `plain_text_to_paragraphs` | plain text → `Paragraph` list | §1 plain text adapter |
| `chuncking.py` | `paginate_semantic` | split into chunks at the lowest-similarity seam | §1 pagination (mock/real API), §1 embedding-failure fallback |
| `chuncking.py` | `_paginate_by_word_count_only` (internal fallback) | split sequentially by min/max words only, when embedding fails | §1 embedding-failure fallback |
| `chuncking.py` | `chunk_containing_position` | source char position → chunk lookup | §1 position-to-chunk lookup |
| `embeddings.py` | `embed_texts` | one raw NIM embedding API call | §3 real NIM embedding API |
| `embeddings.py` | `embed_with_retry` | embedding call with retries | §2 direct low-level embedding utility calls |
| `embeddings.py` | `cosine_similarity` | cosine similarity | §2 direct low-level embedding utility calls |
| `embeddings.py` | `load_config` | load a YAML config | §1 throughout (loading chunk/embedding settings) |
| `embeddings.py` | `EmbeddingAPIError` | embedding API failure exception | §1 embedding-failure fallback |
| `gisting.py` | `embed_chunks` | batch+retry embedding for a chunk list | §2 batch chunk embedding |
| `gisting.py` | `split_into_sentences` | text → sentence list | §2 sentence splitting |
| `gisting.py` | `score_chunk_sentences` | per-sentence importance score *within* a chunk | §2 intra-chunk importance scores |

## 1. Chunking (`chuncking.py`)

Cut priority: ① never split inside a sentence (nltk) → ② keep a dialogue
group with unclosed quotes together as one unit → ③ if a dialogue group
exceeds max_words, fall back to sentence-level splitting → ④ `<p>` boundaries
are candidate points only → ⑤ only a structural-break paragraph is a hard
boundary → ⑥ among seam candidates in the min_words~max_words range, pick
the one with the lowest embedding similarity as the chunk boundary
(`paginate_semantic`).

### Plain text adapter (`plain_text_to_paragraphs`)

The source text (`data/sample/pipeline_test_sample.txt` — the first ~9,900
words of the same public-domain narrativeqa document
`01_length_stress_prep.ipynb` uses, committed here so this notebook has no
upstream dependency) is plain text with paragraphs separated by blank
lines (`\n\s*\n+`) — the real production input format. A single newline
inside a block is a line wrap, not a paragraph boundary, so it's normalized
to a space. A paragraph made entirely of structural-break markers is
flagged `is_scene_break` and becomes a hard boundary during pagination.

In [47]:
import yaml

from src.pipeline.chuncking import (
    chunk_containing_position,
    paginate_semantic,
    plain_text_to_paragraphs,
)
from src.pipeline.embeddings import load_config

with open("data/sample/pipeline_test_sample.txt", encoding="utf-8") as f:
    raw = f.read()

paragraphs = plain_text_to_paragraphs(raw)
print("paragraph count:", len(paragraphs), "\n")
for p in paragraphs:
    flag = " [scene_break]" if p.is_scene_break else ""
    print(f"[{p.index:2d}] offset={p.char_offset:5d} words={len(p.text.split()):4d}{flag} | {p.text[:40]}")

paragraph count: 192 

[ 0] offset=    0 words=   5 | A Thief in the Night
[ 1] offset=   22 words=   5 | [A Book of Raffles' Adventures]
[ 2] offset=   56 words=   1 | by
[ 3] offset=   60 words=   3 | E. W. Hornung
[ 4] offset=   78 words=   1 | Contents
[ 5] offset=   90 words=  33 | Out of Paradise The Chest of Silver The 
[ 6] offset=  290 words=   3 | Out of Paradise
[ 7] offset=  307 words= 170 | If I must tell more tales of Raffles, I 
[ 8] offset= 1197 words= 139 | I pick my words with care and pain, loya
[ 9] offset= 1937 words= 205 | Suffice it that I had been engaged to he
[10] offset= 2973 words=  61 | "We must dine and celebrate the rare eve
[11] offset= 3270 words= 185 | And at the CafÃ© Royal I incontinently t
[12] offset= 4253 words=  23 | "Hector Carruthers!" murmured Raffles, r
[13] offset= 4398 words=  41 | "Not a thing for ages," I replied. "I wa
[14] offset= 4598 words=   7 | And I laughed bitterly in my glass.
[15] offset= 4635 words=  11 | "Nice house?" said Raf

### Pagination (`paginate_semantic`) — mock embeddings

`configs/chunking.yaml`'s min/max_words set the lower/upper bound of the
seam-search window. Flow first, with mock embeddings; the real-API demo is
at the end of this section.

In [48]:
chunk_cfg = yaml.safe_load(open("configs/chunking.yaml", encoding="utf-8"))
gist_cfg = load_config("configs/importance_filter.yaml")

mock_cfg = dict(gist_cfg)
mock_cfg["api_key_env"] = "MOCK_EMBED_KEY"
os.environ["MOCK_EMBED_KEY"] = "offline-dummy"


def fake_embed(texts, input_type, model, truncate, api_key, timeout=30.0, dimensions=None):
    """Fake vectors hashed from paragraph text — no real meaning, but they differ
    per paragraph so seam similarity actually varies."""
    return [[hash(t) % 97 / 97.0, hash(t[::-1]) % 89 / 89.0, 0.1] for t in texts]


chunks = paginate_semantic(
    paragraphs,
    min_words=chunk_cfg["min_words"],
    max_words=chunk_cfg["max_words"],
    granularity="paragraph",
    config=mock_cfg,
    embed_fn=fake_embed,
)

print(f"chunk count: {len(chunks)}  (min_words={chunk_cfg['min_words']}, max_words={chunk_cfg['max_words']})\n")
for c in chunks:
    print(
        f"[{c.index:2d}] words={len(c.text.split()):3d} "
        f"chars=({c.char_start:5d}~{c.char_end:5d}) paragraphs={c.paragraph_indices} | {c.text[:35]}"
    )

chunk count: 38  (min_words=150, max_words=350)

[ 0] words=221 chars=(    0~ 1188) paragraphs=[0, 1, 2, 3, 4, 5, 6, 7] | A Thief in the Night [A Book of Raf
[ 1] words=344 chars=( 1197~ 2964) paragraphs=[8, 9] | I pick my words with care and pain,
[ 2] words=328 chars=( 2973~ 4712) paragraphs=[10, 11, 12, 13, 14, 15] | "We must dine and celebrate the rar
[ 3] words=347 chars=( 4714~ 6591) paragraphs=[16, 17, 18, 19, 20, 21, 22, 23, 24, 25] | "Top shelf," said I. "You know the 
[ 4] words=128 chars=( 6594~ 7287) paragraphs=[26] | "In answer to your first question--
[ 5] words=252 chars=( 7292~ 8636) paragraphs=[27] | As it happened, I could, since I kn
[ 6] words=237 chars=( 8646~ 9895) paragraphs=[28, 29, 30, 31, 32, 33, 34, 35] | "It was rather clever of you to not
[ 7] words=216 chars=( 9900~11019) paragraphs=[36, 37, 38, 39, 40, 41, 42, 43] | "I really think you had better stay
[ 8] words=194 chars=(11021~12019) paragraphs=[44, 45, 46, 47, 48, 49] | "No, I sent it to the country. T

### Confirm dialogue doesn't get split at a chunk boundary

Scans every chunk to check whether `paginate_semantic`'s internal
`_dialogue_groups` actually keeps sentences bound by quotation marks
(`“” ‘’ "" ''`) inside a single chunk. Apostrophes (`I'm`, `don't`) are
excluded so they aren't miscounted as quote marks.

In [49]:
import re


def _quotes_balanced(text: str) -> bool:
    stripped = re.sub(r"(?<=[A-Za-z0-9])[’\']", "", text)  # exclude apostrophes
    return stripped.count("‘") == stripped.count("’") and stripped.count("“") == stripped.count("”")


unbalanced = [c.index for c in chunks if not _quotes_balanced(c.text)]
print("chunks ending with an unclosed quote:", unbalanced if unbalanced else "none ✅")

chunks ending with an unclosed quote: none ✅


### Source position → chunk lookup (`chunk_containing_position`)

Used for post-hoc analysis of which chunk a piece of evidence actually
landed in.

In [50]:
needle = "That safe was in its turn so ingeniously hidden that I never should have found it for myself."  # a literal substring to search for in the source text
pos = raw.find(needle)
hit = chunk_containing_position(chunks, pos)
if pos == -1:
    fallback = "That safe was in its turn so ingeniously hidden"
    pos = raw.find(fallback)
    needle = fallback if pos != -1 else needle
    if pos != -1:
        hit = chunk_containing_position(chunks, pos)

if pos == -1 or hit is None:
    raise ValueError(f"Could not find the search string ({needle!r}) in raw, or no chunk covers that position.")
print(f"source char {pos} (where {needle!r} appears) -> chunk {hit.index}")
print("start of chunk:", hit.text[:80])

# Sanity check: does the source lookup actually match?
print("source text at that position:", raw[hit.char_start : hit.char_start + 80].replace("\n", " "))

# Out-of-range position should return None
print("out-of-range lookup result:", chunk_containing_position(chunks, 10**9))

source char 7997 (where 'That safe was in its turn so ingeniously hidden' appears) -> chunk 5
start of chunk: As it happened, I could, since I knew from his niece that it was one on which Mr
source text at that position: As it happened, I could, since I knew from his niece that it was one on which Mr
out-of-range lookup result: None


### Pagination — real API (`paginate_semantic`)

Only uses the first few paragraphs, to save quota.

In [51]:
if not API_KEY_SET:
    print("No NVIDIA_NIM_API_KEY — skipping.")
else:
    sample_paragraphs = paragraphs[:8]  # save quota
    chunks_real = paginate_semantic(
        sample_paragraphs,
        min_words=chunk_cfg["min_words"],
        max_words=chunk_cfg["max_words"],
        granularity="paragraph",
        config=gist_cfg,
    )
    print(f"{len(sample_paragraphs)} input paragraphs -> {len(chunks_real)} chunks\n")
    for c in chunks_real:
        print(f"[{c.index}] words={len(c.text.split())} paragraphs={c.paragraph_indices} | {c.text[:50]}")

8 input paragraphs -> 1 chunks

[0] words=221 paragraphs=[0, 1, 2, 3, 4, 5, 6, 7] | A Thief in the Night [A Book of Raffles' Adventure


### Fallback on embedding failure (`_paginate_by_word_count_only`)

If `paginate_semantic` keeps failing to reach the embedding API, behavior
depends on `config["on_error"]` — `pass_through` fully replaces it with
`_paginate_by_word_count_only` (the internal fallback that just fills
sequentially by min/max words, formerly `paginate_fixed`), with no
similarity judgment at all; `raise` propagates `EmbeddingAPIError` as-is.

In [52]:
from src.pipeline.chuncking import _paginate_by_word_count_only
from src.pipeline.embeddings import EmbeddingAPIError


def always_fails(*args, **kwargs):
    raise EmbeddingAPIError("simulated failure")


fail_cfg = dict(mock_cfg)
fail_cfg["max_retries"] = 0
fail_cfg["on_error"] = "pass_through"

fallback_chunks = paginate_semantic(
    paragraphs,
    min_words=chunk_cfg["min_words"],
    max_words=chunk_cfg["max_words"],
    granularity="paragraph",
    config=fail_cfg,
    embed_fn=always_fails,
)
direct_chunks = _paginate_by_word_count_only(paragraphs, chunk_cfg["min_words"], chunk_cfg["max_words"])

print("pass_through fallback chunk count:", len(fallback_chunks), "vs. direct _paginate_by_word_count_only call:", len(direct_chunks))
print("text is fully identical:", [c.text for c in fallback_chunks] == [c.text for c in direct_chunks])

# Confirm on_error="raise" actually raises EmbeddingAPIError
fail_cfg_raise = dict(fail_cfg)
fail_cfg_raise["on_error"] = "raise"
try:
    paginate_semantic(
        paragraphs,
        min_words=chunk_cfg["min_words"],
        max_words=chunk_cfg["max_words"],
        granularity="paragraph",
        config=fail_cfg_raise,
        embed_fn=always_fails,
    )
    print("\nEmbeddingAPIError was NOT raised — problem!")
except EmbeddingAPIError as e:
    print("\nEmbeddingAPIError raised as expected:", e)

[embeddings] 임베딩 API 호출 실패 (재시도 0회 소진, 텍스트 16개, input_type='passage'): EmbeddingAPIError: simulated failure


pass_through fallback chunk count: 33 vs. direct _paginate_by_word_count_only call: 33
text is fully identical: True

EmbeddingAPIError raised as expected: 임베딩 API 호출이 재시도 후에도 실패해 의미 기반 페이지네이션을 진행할 수 없습니다. 원인: EmbeddingAPIError: simulated failure


[embeddings] 임베딩 API 호출 실패 (재시도 0회 소진, 텍스트 16개, input_type='passage'): EmbeddingAPIError: simulated failure


## 2. Embedding/score preprocessing (`embeddings.py` + `gisting.py`) — offline, mock embeddings

- `embed_with_retry` / `cosine_similarity` — low-level embedding utilities
  (embeddings.py), called directly.
- `embed_chunks` — batch-embed a chunk list (gisting.py).
- `split_into_sentences` — split text into a sentence list (gisting.py).
- `score_chunk_sentences` — per-sentence importance score *within* a chunk
  (cosine similarity between the chunk's own embedding and each sentence's
  embedding). Drops nothing, just returns scores.

Chunk-question similarity (`score_chunks`, `embed_query`, `ScoredChunk`) is
out of scope here — removed from gisting.py, no mock or real API demo.

Actual compression (chunk summarization) will be added as a separate module
when rehearsal is implemented.

### Direct low-level embedding utility calls (`embed_with_retry`, `cosine_similarity`)

Embeds text directly with `embed_with_retry`, then computes
`cosine_similarity` between the resulting vectors. Checks that
similar-topic sentences score higher than dissimilar ones.

In [53]:
from src.pipeline.embeddings import cosine_similarity, embed_with_retry


def fake_embed_by_keyword(texts, input_type, model, truncate, api_key, timeout=30.0, dimensions=None):
    """Fake embedding that splits vectors by whether "cat" is present — makes the
    similarity difference easy to see."""
    return [[1.0 if "cat" in t else 0.0, 1.0 if "cat" not in t else 0.0, 0.1] for t in texts]


sample_texts = ["The cat looks out the window.", "The cat takes a nap.", "The stock index plunged."]
vectors = embed_with_retry(sample_texts, "passage", mock_cfg, fake_embed_by_keyword)

print("vectors returned:", len(vectors))
for t, v in zip(sample_texts, vectors):
    print(f"  {v} | {t}")

sim_similar = cosine_similarity(vectors[0], vectors[1])  # both cat-related
sim_different = cosine_similarity(vectors[0], vectors[2])  # different topic

print(f"\nsimilarity between the two 'cat' sentences: {sim_similar:.4f} (high)")
print(f"similarity with the unrelated sentence: {sim_different:.4f} (low)")

vectors returned: 3
  [1.0, 0.0, 0.1] | The cat looks out the window.
  [1.0, 0.0, 0.1] | The cat takes a nap.
  [0.0, 1.0, 0.1] | The stock index plunged.

similarity between the two 'cat' sentences: 1.0000 (high)
similarity with the unrelated sentence: 0.0099 (low)


### Batch chunk embedding (`embed_chunks`)

Passes the whole chunk list to `embed_chunks` at once to show how
batching+retry behaves. One of the results is reused directly as the
`chunk_embedding` argument in the `score_chunk_sentences` demo below.

In [54]:
from src.pipeline.gisting import embed_chunks

chunk_embeddings = embed_chunks(chunks, mock_cfg, embed_fn=fake_embed)
print(f"{len(chunks)} chunks -> {len(chunk_embeddings)} embeddings (batch_size={mock_cfg['batch_size']})")
print("first chunk's embedding:", chunk_embeddings[0])

38 chunks -> 38 embeddings (batch_size=16)
first chunk's embedding: [0.5360824742268041, 0.9438202247191011, 0.1]


### Sentence splitting (`split_into_sentences`)

Feeds one chunk's `text` directly into `split_into_sentences` and checks
the resulting sentence list.

In [55]:
from src.pipeline.gisting import split_into_sentences

sample_sentences = split_into_sentences(chunks[0].text)
print(f"chunk 0 split into {len(sample_sentences)} sentence(s):\n")
for s in sample_sentences[:5]:
    print(" -", s[:60])

chunk 0 split into 1 sentence(s):

 - A Thief in the Night [A Book of Raffles' Adventures] by E. W


### Intra-chunk importance scores (`score_chunk_sentences`)

Shows, as scores only, which sentences within a chunk look "important" — no
compression happens here.

In [56]:
from src.pipeline.gisting import score_chunk_sentences

sample_chunk = chunks[0]
chunk_embedding = chunk_embeddings[sample_chunk.index]  # reuse the embed_chunks result above as-is
scored = score_chunk_sentences(sample_chunk, chunk_embedding, config=mock_cfg, embed_fn=fake_embed)

print(f"chunk {sample_chunk.index} per-sentence importance (top 5):\n")
for sentence, score in sorted(scored, key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {score:.3f} | {sentence[:60]}")

chunk 0 per-sentence importance (top 5):

  1.000 | A Thief in the Night [A Book of Raffles' Adventures] by E. W


## 3. Real NIM embedding API

Only runs if `NVIDIA_NIM_API_KEY` is set. Uses only a few early chunks to
save quota.

In [57]:
if not API_KEY_SET:
    print("No NVIDIA_NIM_API_KEY — skipping.")
else:
    from src.pipeline.embeddings import embed_texts

    emb = embed_texts(
        ["This is a test sentence."],
        input_type="passage",
        model=gist_cfg["model"],
        truncate=gist_cfg["truncate"],
        api_key=os.environ["NVIDIA_NIM_API_KEY"],
    )
    print("model:", gist_cfg["model"])
    print("embedding dimensions:", len(emb[0]), "| first 5 values:", [round(v, 4) for v in emb[0][:5]])

model: nvidia/llama-nemotron-embed-1b-v2
embedding dimensions: 2048 | first 5 values: [-0.0068, -0.014, 0.0282, 0.0119, 0.0305]
